# Lab 6: Hubs, Centrality, and Preferential Attachment

In this lab, we combine three themes from network science into one coherent workflow:

- degree structure and hubs,
- competing notions of centrality,
- growth mechanisms that generate hub-heavy networks.

The goal is not just to compute statistics, but to compare different ways of describing importance and to ask which network models better capture the structure of a realistic hub-heavy network.

In [ ]:
from lab6_helpers import *
import pandas as pd

### Part 1: First look at degree structure

We begin with a small graph and study degree as both a local and global property.

In [ ]:
G = sample_graph()
graph_summary(G)

In [ ]:
draw_graph(G, "Sample graph", with_labels=True)

In [ ]:
degree_table(G)

In [ ]:
degree_sequence(G)

In [ ]:
plot_degree_histogram(G, "Degree histogram for sample graph")

Questions:
- Which vertex has the largest degree?
- Does the graph appear to contain a hub?
- What does the degree histogram show that is harder to see immediately in the picture?

### Part 2: Compare basic graph families

Now compare a path graph, a cycle graph, and a star graph. These graphs have the same general size, but very different degree structure.

In [ ]:
P = path_graph(10)
C = cycle_graph(10)
S = star_graph(10)

In [ ]:
draw_graph(P, "Path graph", with_labels=False)
draw_graph(C, "Cycle graph", with_labels=False)
draw_graph(S, "Star graph", with_labels=False)

In [ ]:
degree_sequence(P), average_degree(P)

In [ ]:
degree_sequence(C), average_degree(C)

In [ ]:
degree_sequence(S), average_degree(S)

In [ ]:
plot_degree_histogram(P, "Path graph degree histogram")
plot_degree_histogram(C, "Cycle graph degree histogram")
plot_degree_histogram(S, "Star graph degree histogram")

Questions:
- Which graph has the strongest hub?
- Which graph has the most uniform degree structure?
- Why is average degree not enough to distinguish these graphs?

### Part 3: Degree CDF and CCDF

Histograms are useful, but cumulative views often show the upper tail more clearly.

In [ ]:
plot_degree_cdf(S, "Star graph degree CDF")
plot_degree_ccdf(S, "Star graph degree CCDF")

In [ ]:
R = random_graph(100, 196, seed=21)
B = ba_graph(100, 2, seed=21)

In [ ]:
plot_degree_ccdf(R, "ER graph degree CCDF")
plot_degree_ccdf(B, "BA graph degree CCDF")
plot_degree_ccdf_loglog(R, "ER graph CCDF (log-log)")
plot_degree_ccdf_loglog(B, "BA graph CCDF (log-log)")

Questions:
- Which graph appears to have the heavier tail?
- What does the CCDF show more clearly than the histogram?
- Why might a log-log view be useful when studying hubs?

### Part 4: Centrality as competing notions of importance

Now we move from degree to centrality. Different centrality measures capture different ideas of what it means for a vertex to be important.

In [ ]:
Bottleneck = bridge_graph()
HubBridge = hub_bridge_graph()

In [ ]:
draw_graph(Bottleneck, "Bridge graph", with_labels=True)
draw_graph(HubBridge, "Hub-bridge graph", with_labels=True)

In [ ]:
centrality_table(Bottleneck)

In [ ]:
centrality_table(HubBridge)

In [ ]:
compare_rankings(HubBridge)

Questions:
- Which node has the highest degree centrality?
- Which node has the highest betweenness centrality?
- Are they the same?
- What does this tell you about hubs versus bottlenecks?

### Part 5: A real network example

Now apply these ideas to Zachary's Karate Club graph.

In [ ]:
K = karate_graph()
graph_summary(K)

In [ ]:
draw_graph(K, "Karate Club graph", with_labels=True)

In [ ]:
top_k_degrees(K, k=10)

In [ ]:
compare_rankings(K).head(10)

Questions:
- Which vertices rank highly across several measures?
- Which vertices rank highly by one measure but not another?
- Does the graph appear to contain hubs, bridges, or both?

### Part 6: Preferential attachment and hub-heavy structure

We now compare three networks on roughly the same scale:
- an observed Internet AS graph model,
- an ER graph,
- a BA graph.

In [ ]:
AS = internet_as_graph(300, seed=13)
n = AS.number_of_nodes()
m = AS.number_of_edges()

ER = er_graph(n, m, seed=13)
BA = ba_graph(n, 2, seed=13)

In [ ]:
compare_basic_degree_stats([AS, ER, BA], ["Internet AS", "ER", "BA"])

In [ ]:
draw_graph_by_type(AS, "Internet AS graph by node type")
draw_graph_sized_by_degree(ER, "ER graph sized by degree")
draw_graph_sized_by_degree(BA, "BA graph sized by degree")
draw_graph_sized_by_degree(AS, "Internet AS graph sized by degree")

In [ ]:
type_counts(AS)

In [ ]:
top_k_degrees(AS)

Questions:
- Which graph has the most visually prominent hubs?
- Which graph appears most homogeneous?
- Does the Internet AS graph look more like the ER graph or the BA graph?
- Why is the Internet AS graph a better real-world hub-heavy example than a toy star graph?

### Part 7: Statistical comparison of hub structure

Now compare the observed Internet AS graph to ER and BA null models using hub-sensitive statistics.

In [ ]:
observed_stats = observed_hub_statistics(AS)
observed_stats

In [ ]:
m_ba = max(1, round(m / n))
m_ba

In [ ]:
er_null = simulate_er_hub_statistics(n, m, trials=300, seed=21)
ba_null = simulate_ba_hub_statistics(n, m_ba, trials=300, seed=21)

In [ ]:
summarize_null(er_null["max_degree"]), summarize_null(ba_null["max_degree"])

In [ ]:
plot_null_histogram(
    er_null["max_degree"],
    observed_value=observed_stats["max_degree"],
    xlabel="Maximum degree",
    title="ER null distribution of maximum degree"
)
plot_null_histogram(
    ba_null["max_degree"],
    observed_value=observed_stats["max_degree"],
    xlabel="Maximum degree",
    title="BA null distribution of maximum degree"
)

In [ ]:
p_er_max = empirical_p_value_upper(er_null["max_degree"], observed_stats["max_degree"])
p_ba_max = empirical_p_value_upper(ba_null["max_degree"], observed_stats["max_degree"])
p_er_max, p_ba_max

In [ ]:
plot_null_histogram(
    er_null["degree_gini"],
    observed_value=observed_stats["degree_gini"],
    xlabel="Degree Gini coefficient",
    title="ER null distribution of degree inequality"
)
plot_null_histogram(
    ba_null["degree_gini"],
    observed_value=observed_stats["degree_gini"],
    xlabel="Degree Gini coefficient",
    title="BA null distribution of degree inequality"
)

In [ ]:
p_er_gini = empirical_p_value_upper(er_null["degree_gini"], observed_stats["degree_gini"])
p_ba_gini = empirical_p_value_upper(ba_null["degree_gini"], observed_stats["degree_gini"])
pd.DataFrame({
    "statistic": ["max_degree", "degree_gini"],
    "p_value_ER": [p_er_max, p_er_gini],
    "p_value_BA": [p_ba_max, p_ba_gini],
})

Questions:
- Relative to which null model does the observed graph look more unusual?
- Does the BA model fit the observed hub structure better than the ER model?
- Why does a better fit to BA not prove preferential attachment generated the observed network?

### Part 8: Gephi exploration

Export a few graphs to Gephi and compare what the visualizations suggest.

In [ ]:
export_for_gephi(K, "lab06_karate_graph.gexf")
export_for_gephi(HubBridge, "lab06_hub_bridge_graph.gexf")
export_for_gephi(AS, "lab06_internet_as_graph.gexf")
export_for_gephi(ER, "lab06_er_graph.gexf")
export_for_gephi(BA, "lab06_ba_graph.gexf")

In Gephi:

1. Import the karate, ER, BA, and Internet AS graphs.
2. Apply the same layout to each.
3. Size nodes by degree.
4. Compare which graphs are most hub-heavy and which appear more uniform.

Questions:
- Which graph makes hubs stand out most strongly?
- Which graph looks most homogeneous?
- Does the visual impression match the histograms and null-model comparisons?
- What can the visualization suggest, and what still requires a statistic?

### Part 9: Reflection

Answer the following in complete sentences.

1. What is a hub?
2. Why is average degree not enough to describe hub-heavy networks?
3. What does a degree CCDF show that a histogram may hide?
4. Why can two centrality measures disagree about which vertex is most important?
5. What is the difference between a hub and a bottleneck?
6. Why does preferential attachment generate hub-heavy graphs?
7. Why is similarity to the BA model not enough to prove a mechanism?
8. What observation from this lab feels most like the start of a research question?